In [53]:
import uuid
import os
import re
import shutil
from pathlib import Path

## Configs

In [54]:
BASE_DIR = Path("./teste_bucket/profile-documents")

# Exemplo: 0003ecb1-1835-445a-9b4f-59138-2024-04-09.jpg -> 0003ecb1-1835-445a-9b4f-59138...
UUID_PATTERN = re.compile(
    r"^([0-9a-fA-F]{8}-[0-9a-fA-F]{4}-[0-9a-fA-F]{4}-[0-9a-fA-F]{4}-[0-9a-fA-F]{12})"
)

## Methods

In [ ]:
def extrair_id_pasta(nome_arquivo: str) -> str | None:
    """Extrai o UUID correspondente ao nome da pasta."""
    match = UUID_PATTERN.match(nome_arquivo)
    if match:
        return match.group(1)

    partes = nome_arquivo.split("-")
    if len(partes) >= 5:
        possivel_id = "-".join(partes[:5])
        return possivel_id

    return None


def criar_cenario_teste():
    if BASE_DIR.exists():
        shutil.rmtree(BASE_DIR.parent)

    BASE_DIR.mkdir(parents=True, exist_ok=True)

    id_1 = "00076ac7-fbc2-42f3-9620-8cb4f19bca5e"
    (BASE_DIR / id_1).mkdir(exist_ok=True)
    (BASE_DIR / id_1 / "avatar.png").touch()
    (BASE_DIR / f"{id_1}-2024-05-15.jpg").touch()

    id_2 = "0003ecb1-1835-445a-9b4f-59138d612345"
    (BASE_DIR / f"{id_2}-2024-04-09.jpg").touch()

    id_3 = "0008b6b4-447a-4bba-b1ca-09a7fed7ced8"
    (BASE_DIR / f"{id_3}.pdf").touch()

    id_4 = "0009c7c5-558b-5ccb-c2db-10b8gfe8dfe9"
    (BASE_DIR / f"{id_4}.png").touch()

    id_5 = "0010d8d6-669c-6ddc-d3ec-21c9hgf9egfa"
    (BASE_DIR / f"{id_5}-2023-12-01.jpeg").touch()

    id_6 = "aaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaa"
    (BASE_DIR / id_6).mkdir(exist_ok=True)

    print("Ambiente de teste criado com sucesso em:", BASE_DIR.resolve())
    print("Arquivos e pastas iniciais:")
    for item in sorted(BASE_DIR.iterdir()):
        print(f" - [{'PASTA' if item.is_dir() else 'ARQUIVO'}] {item.name}")

def organizar_diretorio_local(dry_run: bool = False):
    """Varre os arquivos soltos e move para suas respectivas pastas com o novo formato."""
    print(f"\nIniciando organização {'(DRY-RUN)' if dry_run else ''}...")

    itens = list(BASE_DIR.iterdir())
    arquivos_soltos = [item for item in itens if item.is_file()]

    if not arquivos_soltos:
        print("Nenhum arquivo solto encontrado para organizar.")
        return

    # Padrão para encontrar data no formato YYYY-MM-DD
    DATE_PATTERN = re.compile(r'-?\d{4}-\d{2}-\d{2}')

    for arquivo in arquivos_soltos:
        pasta_alvo_nome = extrair_id_pasta(arquivo.name)

        if not pasta_alvo_nome:
            print(f"Não foi possível identificar o ID de: {arquivo.name}")
            continue

        pasta_destino = BASE_DIR / pasta_alvo_nome

        extensao = arquivo.suffix
        nome_sem_extensao = arquivo.stem

        # Verifica se tem data e remove
        if DATE_PATTERN.search(nome_sem_extensao):
            nome_sem_extensao = DATE_PATTERN.sub('', nome_sem_extensao)
            tinha_data = True
        else:
            tinha_data = False

        novo_uuid = str(uuid.uuid4())
        novo_nome_arquivo = f"{nome_sem_extensao}_{novo_uuid}{extensao}"

        destino_final = pasta_destino / novo_nome_arquivo

        if dry_run:
            print(f"[SIMULAÇÃO] Mover: {arquivo.name} -> {pasta_alvo_nome}/{novo_nome_arquivo}")
            if not tinha_data:
                print(f"  -> AVISO: O arquivo '{arquivo.name}' foi transferido sem data!")
        else:
            try:
                pasta_destino.mkdir(parents=True, exist_ok=True)
                shutil.move(str(arquivo), str(destino_final))
                print(f"Movido: {arquivo.name} -> {pasta_alvo_nome}/{novo_nome_arquivo}")
                if not tinha_data:
                    print(f"AVISO: O arquivo '{arquivo.name}' não tinha data!")
            except Exception as e:
                print(f"\n[ERRO CRÍTICO] Falha ao transferir o arquivo {arquivo.name}!")
                print(f"Detalhes do erro: {e}")
                print("Interrompendo a execução imediatamente.")
                break

    print("\nOrganização finalizada!")


In [56]:
criar_cenario_teste()

Ambiente de teste criado com sucesso em: /home/enzo-tabuchi/Documentos/Instituto/GCP/teste_bucket/profile-documents
Arquivos e pastas iniciais:
 - [ARQUIVO] 0003ecb1-1835-445a-9b4f-59138d612345-2024-04-09.jpg
 - [PASTA] 00076ac7-fbc2-42f3-9620-8cb4f19bca5e
 - [ARQUIVO] 00076ac7-fbc2-42f3-9620-8cb4f19bca5e-2024-05-15.jpg
 - [ARQUIVO] 0008b6b4-447a-4bba-b1ca-09a7fed7ced8.pdf
 - [ARQUIVO] 0009c7c5-558b-5ccb-c2db-10b8gfe8dfe9.png
 - [ARQUIVO] 0010d8d6-669c-6ddc-d3ec-21c9hgf9egfa-2023-12-01.jpeg
 - [PASTA] aaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaa


In [57]:
organizar_diretorio_local(dry_run=True)


Iniciando organização (DRY-RUN)...
[SIMULAÇÃO] Mover: 0008b6b4-447a-4bba-b1ca-09a7fed7ced8.pdf -> 0008b6b4-447a-4bba-b1ca-09a7fed7ced8/0008b6b4-447a-4bba-b1ca-09a7fed7ced8_e56bf059-3908-4dc8-9921-50364914dc9f.pdf
  -> AVISO: O arquivo '0008b6b4-447a-4bba-b1ca-09a7fed7ced8.pdf' foi transferido sem data!
[SIMULAÇÃO] Mover: 0010d8d6-669c-6ddc-d3ec-21c9hgf9egfa-2023-12-01.jpeg -> 0010d8d6-669c-6ddc-d3ec-21c9hgf9egfa/0010d8d6-669c-6ddc-d3ec-21c9hgf9egfa_1b9a20a8-5e55-4528-b4e6-6b907b03a66a.jpeg
[SIMULAÇÃO] Mover: 0003ecb1-1835-445a-9b4f-59138d612345-2024-04-09.jpg -> 0003ecb1-1835-445a-9b4f-59138d612345/0003ecb1-1835-445a-9b4f-59138d612345_d358e552-ce58-4510-9c79-f4c715b8b1db.jpg
[SIMULAÇÃO] Mover: 00076ac7-fbc2-42f3-9620-8cb4f19bca5e-2024-05-15.jpg -> 00076ac7-fbc2-42f3-9620-8cb4f19bca5e/00076ac7-fbc2-42f3-9620-8cb4f19bca5e_a8d0c191-c28d-4f4f-a757-64266902becd.jpg
[SIMULAÇÃO] Mover: 0009c7c5-558b-5ccb-c2db-10b8gfe8dfe9.png -> 0009c7c5-558b-5ccb-c2db-10b8gfe8dfe9.png/0009c7c5-558b-5ccb-c2d

In [58]:
# organizar_diretorio_local(dry_run=False)

In [59]:
# print("\nEstrutura final resultante:")
# for item in sorted(BASE_DIR.iterdir()):
#     if item.is_dir():
#         conteudo = [f.name for f in item.iterdir()]
#         print(f" {item.name}/ ({len(conteudo)} itens: {conteudo})")